In [ ]:
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve().parent

In [ ]:
import rosbags
from rosbags.highlevel import AnyReader

# Load message definitions.
BAG_PATH = Path("../data/euroc/MH_01_easy.bag").resolve()
with AnyReader([BAG_PATH]) as reader:
    pass

from rosbags.usertypes import (
    sensor_msgs__msg__Image as ImageMessage,
    sensor_msgs__msg__Imu as ImuMessage,
)
from scipy.spatial.transform import Rotation

In [ ]:
def to_array(data: Any) -> List[int | float]:
    d = data.__dict__
    d.pop("__msgtype__")
    return [round(v, 2) for v in d.values()]


def decode_imu_data(message: ImuMessage) -> Dict[str, List[int | float]]:
    acceleration = to_array(message.linear_acceleration)

    q = message.orientation
    orientation = Rotation.from_quat((q.x, q.y, q.z, q.w)).as_euler("xyz")
    angular_rate = to_array(message.angular_velocity)

    return {
        "acceleration": acceleration,
        "orientation": orientation,
        "angular_rate": angular_rate,
    }

In [ ]:
import cv2


def decode_camera_data(message: ImageMessage) -> np.ndarray:
    image = np.frombuffer(message.data, dtype=np.uint8)
    image = image.reshape(message.height, message.width)
    return image


def display_video(images: List[np.ndarray], output_filepath="output.mp4") -> None:
    h, w = images[0].shape
    out = cv2.VideoWriter(
        output_filepath,
        cv2.VideoWriter_fourcc(*"mp4v"),
        20.0,  # fps
        (w, h),
        isColor=False,
    )

    for image in images:
        out.write(image)
    out.release()

    print('frame count: ', int(cv2.VideoCapture(output_filepath).get(cv2.CAP_PROP_FRAME_COUNT)))

In [ ]:
TARGET_TOPICS = [
    "/imu0",
    "/cam0/image_raw",
]


def print_topic(connection: rosbags.interfaces.Connection) -> None:
    print(
        f"id: {connection.id}, "
        f"topic: {connection.topic}, "
        f"msgtype: {connection.msgtype}, "
        f"count: {connection.msgcount}"
    )


def read_messages(bag_path: Path):
    with AnyReader([bag_path]) as reader:
        for connection in reader.connections:
            print_topic(connection)
        print()

        imu_outputs = []
        images = []
        connections = [c for c in reader.connections if c.topic in TARGET_TOPICS]
        for i, (connection, timestamp, rawdata) in enumerate(reader.messages(connections=connections)):
            message = reader.deserialize(rawdata, connection.msgtype)
            if connection.topic == "/imu0":
                imu_outputs.append(decode_imu_data(message))
            elif connection.topic == "/cam0/image_raw":
                images.append(decode_camera_data(message))

        display(pd.DataFrame(imu_outputs))
        display_video(images)


read_messages(BAG_PATH)